# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR\u02c6\u00b2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. The dataset describes ordered logistic regression outputs related to adoption predictors in rangeland management practices across Northern Kenya.

### Dataset Source
The dataset source is a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`. This allows us to inspect the structure and content defined by the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View basic metadata
meta = dataset.metadata
print(f"Dataset name: {getattr(meta, 'name', None)}\n\nDescription: {getattr(meta, 'description', None)}")

## 2. Data Overview
Let's examine the record sets defined in the Croissant schema, as well as available fields and their unique `@id`s.

Entities are always referenced by their `@id`.

In [ ]:
# List available record sets and their fields by @id
record_sets = dataset.metadata.record_sets
for rs in record_sets:
    print(f"Record set name: {rs.name}\n  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (id: {field.id}) type: {field.data_type}")
    print()
# Store available record set @ids for future use
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Load the available record sets into pandas DataFrames using their `@id`. This enables further exploration and manipulation using pandas.

We use the variable `record_set_ids` that contains all record set `@id`s detected in the previous cell.

In [ ]:
# Load data from each record set to DataFrames, keyed by @id
dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet {rs_id} ...")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# As an illustration, print fields from one record set
if record_set_ids:
    example_rs_id = record_set_ids[0]
    print(f"\nFields (columns) for record set '@id': {example_rs_id}")
    print(dataframes[example_rs_id].columns.tolist())
    display(dataframes[example_rs_id].head())
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Let's filter, normalize, and group the data for analysis. 

We first select one numeric field (by `@id`) from the first record set for demonstration. If your dataset contains multiple record sets or fields, you should repeat or adapt this analysis as needed.

**Note:** Edit the field `@id`s below based on your dataset overview (Section 2) results. For this notebook, field IDs are accessed programmatically.

In [ ]:
# Pick the first record set and one numeric field for demo.
# Adjust these variables according to your schema. This auto-selects the first float/integer field.
import numpy as np

selected_rs_id = None
numeric_field_id = None
group_field_id = None

for rs in dataset.metadata.record_sets:
    for field in rs.fields:
        if field.data_type in ('Float', 'Integer', 'Number', 'schema:Float', 'schema:Integer', 'schema:Number'):
            selected_rs_id = rs.id
            numeric_field_id = field.id
            break
    if selected_rs_id:
        # Optionally select a first String/Text field as grouping
        for field in rs.fields:
            if field.data_type.lower().find('text') > -1 or field.data_type.lower().find('string') > -1:
                group_field_id = field.id
                break
        break

if selected_rs_id and numeric_field_id:
    df = dataframes[selected_rs_id]
    # Some columns may not be numeric yet, attempt conversion
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    # Remove records where the field is missing
    filtered_df = df[df[numeric_field_id].notnull()]

    # Example filtering: Use 75th percentile as threshold
    threshold = filtered_df[numeric_field_id].quantile(0.75)
    filtered_df = filtered_df[filtered_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        (filtered_df[numeric_field_id].std() if filtered_df[numeric_field_id].std() != 0 else 1)
    )
    print(f"Normalized {numeric_field_id} for filtered records (first 5):")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(f"Grouped data by '{group_field_id}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No suitable numeric field found in the dataset.")

## 5. Visualization
Visualize the distribution of a selected numeric field from one record set. You can adapt this for other fields or relationships as required.

> **Tip:** If field/column names are not human-friendly (e.g. contain @id URLs), insert a mapping for axis labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_rs_id and numeric_field_id and selected_rs_id in dataframes:
    df = dataframes[selected_rs_id]
    # Drop non-numeric values
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    data = df[numeric_field_id].dropna()
    plt.figure(figsize=(8,4))
    sns.histplot(data, kde=True, bins=20)
    plt.title(f"Distribution of field {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
else:
    print("No suitable data for visualization.")

## 6. Conclusion
In this notebook, we demonstrated:
- Loading and introspecting a Croissant-based dataset with `mlcroissant`.
- Listing and referencing all record sets, fields, and columns by their `@id`.
- Extracting tabular records and performing basic exploratory analysis such as filtering and normalization.
- Visualizing data distributions of selected numeric fields.

This structure can be expanded for domain-specific analyses, richer visualizations, or machine learning workflows, depending on your use case.